In [1]:
import math
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline





#VALUE CLASS
class Value:
    def __init__(self, data, _children=(),_op='', label=""):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev =  set(_children)
        self._op = _op
        self.label = label
    def __repr__(self):
        return f"Value(data={self.data})"
    
    def __add__(self, other):
        other = other if isinstance (other, Value) else Value(other)
        out = Value(self.data + other.data, (self,other), '+')
        def _backward():
           self.grad += 1.0 * out.grad
           other.grad += 1.0 * out.grad
        out._backward = _backward
        return out

    def __radd__(self, other):
        other = other if isinstance (other, Value) else Value(other)
        out = Value(self.data + other.data, (self,other), '+')
        def _backward():
           self.grad += 1.0 * out.grad
           other.grad += 1.0 * out.grad
        out._backward = _backward
        return out

    def __mul__(self,other):
        other = other if isinstance (other, Value) else Value(other)
        out = Value(self.data * other.data, (self,other), '*')
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out

    def __rmul__(self,other):
       other = other if isinstance (other, Value) else Value(other)
       out = Value(self.data*other.data, (self,other), "*")
       def _backward():
          self.grad += other.data * out.grad
          other.grad += self.data * out.grad
       out._backward = _backward
       return out

    
    def tanh(self):
       x=self.data
       t = (math.exp(2*x) - 1)/(math.exp(2*x) + 1)
       out = Value(t, (self,), 'tanh')

       def _backward():
          self.grad += (1-t**2)*out.grad
       out._backward = _backward
       return out

    def exp(self):
       x=self.data
       out = Value(math.exp(x), (self,), 'exp')
       def _backward():
          self.grad += out.data * out.grad
       out._backward = _backward
       return out
    
    def __truediv__(self,other):
       out=Value(self.data*other**-1, (self,other), '/')
       return out

    def __rtruediv__(self,other):
       out=Value(other*self.data**-1, (self,other), '/')
       return out
    
    def __pow__(self,other):
        assert isinstance(other, (int, float))
        out = Value(self.data**other, (self,), f'**{other}')
        def _backward():
            self.grad += (other * self.data**(other-1)) * out.grad
        out._backward = _backward
        return out

    def __neg__(self):
       return self  * -1

    def __sub__(self, other):
        return self + (-other)

    def __rsub__(self, other):
       return self + (-other)

    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)

        self.grad = 1.0
        for node in reversed(topo):
            node._backward()




#GRAPHVIZ (JUST COPY-PASTE)
from graphviz import Digraph

def trace(root):
  # builds a set of all nodes and edges in a graph
  nodes, edges = set(), set()
  def build(v):
    if v not in nodes:
      nodes.add(v)
      for child in v._prev:
        edges.add((child, v))
        build(child)
  build(root)
  return nodes, edges

def draw_dot(root):
  dot = Digraph(format='svg', graph_attr={'rankdir': 'LR'}) # LR = left to right
  
  nodes, edges = trace(root)
  for n in nodes:
    uid = str(id(n))
    # for any value in the graph, create a rectangular ('record') node for it
    dot.node(name = uid, label = "{ %s | data %.4f | grad %.4f }" % (n.label, n.data, n.grad), shape='record')
    if n._op:
      # if this value is a result of some operation, create an op node for it
      dot.node(name = uid + n._op, label = n._op)
      # and connect this node to it
      dot.edge(uid + n._op, uid)

  for n1, n2 in edges:
    # connect n1 to the op node of n2
    dot.edge(str(id(n1)), str(id(n2)) + n2._op)

  return dot








#NEURAL NETS

class Neuron:
   def __init__(self, nin):
      self.w = [Value(np.random.randn()) for i  in range(nin)]
      self.b = Value(np.random.randn())

   def __call__(self, x):
      act = sum((wi*xi for wi, xi in zip (self.w, x)), self.b)
      out = act.tanh()
      return out

   def parameters(self):
      return self.w + [self.b]
   
class Layer:
   def __init__(self, nin, nout):
      self.neurons = [Neuron(nin) for _ in range(nout)]

   def __call__(self, x):
      out = [n(x) for n in self.neurons]
      return out[0] if len(out) == 1 else out 

   def parameters(self):
      params=[]
      for n in self.neurons:
         ps = n.parameters()
         params.extend(ps)
      return params

class MLP:
   def __init__(self, nin, nouts):
    sz = [nin] + nouts
    self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]

   def __call__(self, x):
        for layer in self.layers:
             x = layer(x)
        return x

   def parameters(self):
         params=[]
         for n in self.layers:
            ps = n.parameters()
            params.extend(ps)
         return params

#LOSS FUNCTION
x = [[1.0, 2.0], [2.0, 3.0], [3.0, 4.0], [4.0, 5.0]]
n = MLP(2, [4, 4, 1])
y = [n(xi) for xi in x]
ypred = [1.0, 2.0, 3.0, 4.0]
loss = sum([(yp-yi)**2 for yi, yp in zip(y, ypred)])
n.parameters()


[Value(data=-0.18740851924271018),
 Value(data=-0.8326622209082539),
 Value(data=0.20305400022783718),
 Value(data=-0.11627139064686402),
 Value(data=-0.6858580938947109),
 Value(data=-1.261636969718209),
 Value(data=-2.263932977831772),
 Value(data=1.1754530063502437),
 Value(data=1.3274512666121876),
 Value(data=1.1144055221023594),
 Value(data=-0.46019090507481836),
 Value(data=-0.8990106181950551),
 Value(data=1.167818581428951),
 Value(data=0.4683063725673727),
 Value(data=0.5844326283025695),
 Value(data=1.2364233966717102),
 Value(data=0.9968350111393788),
 Value(data=0.46343208445736545),
 Value(data=0.3904955557452474),
 Value(data=-0.6119017994493399),
 Value(data=0.7312444490798194),
 Value(data=2.112232990070795),
 Value(data=1.803336292734541),
 Value(data=1.381030852954743),
 Value(data=-2.0960818195461237),
 Value(data=0.29734591724120646),
 Value(data=-0.5231566537779544),
 Value(data=0.7655470653001383),
 Value(data=-0.7202003878695709),
 Value(data=-1.3974147741075513

In [2]:
x = [[1,2],[2,3],[3,4],[5,6]]
n = MLP(2, [4, 4, 1])
ypred = [n(xi) for xi in x]
y = [1.0, -1.0, -1.0, 1.0]
loss = sum([(yi-yp)**2 for yi, yp in zip(y, ypred)])
loss

Value(data=5.633407218198158)

In [3]:
ypred

[Value(data=-0.6354513150558302),
 Value(data=-0.6422534064413215),
 Value(data=-0.6441100311786101),
 Value(data=-0.6444044268882841)]

In [4]:
while loss.data>0.0001:   
    #forward pass
    ypred = [n(xi) for xi in x]
    loss = sum([(yi-yp)**2 for yi, yp in zip(y, ypred)])  

    #0grad
    for p in n.parameters():
        p.grad = 0
    
    #backward pass      
    loss.backward()

    #gradient descent
    for p in n.parameters():
        p.data += -0.01 * p.grad
    print(loss)

Value(data=5.633407218198158)
Value(data=5.138585485589136)
Value(data=4.621376443172105)
Value(data=4.207331433304841)
Value(data=3.9866085901079265)
Value(data=3.9076560184424025)
Value(data=3.8779961729039925)
Value(data=3.8574751968000314)
Value(data=3.8370690048550085)
Value(data=3.815155682919138)
Value(data=3.7912798049337573)
Value(data=3.7651329750649016)
Value(data=3.7364305700453726)
Value(data=3.7049233280088045)
Value(data=3.670443229226061)
Value(data=3.6329754278959263)
Value(data=3.592754229561097)
Value(data=3.5503682649047414)
Value(data=3.5068353123157845)
Value(data=3.463581295079395)
Value(data=3.4222612370072145)
Value(data=3.3844285663776668)
Value(data=3.3511761839134655)
Value(data=3.3229313398561278)
Value(data=3.2994963441610343)
Value(data=3.2802647799507128)
Value(data=3.2644681335735144)
Value(data=3.251354132730934)
Value(data=3.240276818293)
Value(data=3.2307220874329268)
Value(data=3.2222990476438143)
Value(data=3.2147185139575227)
Value(data=3.20776990

KeyboardInterrupt: 

In [12]:
ypred

[Value(data=0.9937277895226306),
 Value(data=-0.9960929094409368),
 Value(data=-0.9944825378416479),
 Value(data=0.9961334557491663)]